![TecNM](assets/encabezado.png)

---

# Machine Learning y Deep Learning
## Unidad 3 · Modelos de Clasificación

Práctica 2 — Regresión Logística: Predicción de Incumplimiento de Préstamos (Lending Club)

> **Facilitador:** Dr. José Gabriel Rodríguez Rivas  
> **Alumno:** Christian Gibran Espituñal Villanueva  
> **No. Control:** 20041243

---

## Introducción

En la primera parte de esta práctica se comprende el contexto del negocio del dataset a utilizar: el **Club de Préstamos Lending Club**.

**LendingClub** es una compañía estadounidense de préstamos *peer-to-peer*, con sede en San Francisco, California. Es la plataforma de préstamos entre pares más grande del mundo. Permite a los prestatarios solicitar préstamos personales no garantizados entre \$1,000 y \$40,000. Los inversionistas seleccionan los préstamos en los que desean invertir según la información proporcionada (monto, calificación, propósito), con una inversión mínima de \$25. Los inversionistas ganan dinero con los intereses; LendingClub cobra tarifas de originación y servicio.

---

## Objetivo

Construir y evaluar modelos de **Regresión Logística** para predecir si un préstamo será pagado (`repaid = 1`) o no pagado (`repaid = 0`), analizando el efecto del balanceo de clases, el escalado de datos, el ajuste del umbral de decisión y la importancia de las variables.

---

## Marco Teórico

### Regresión Logística

La regresión logística toma un conjunto de variables de entrada $\mathbf{x}$ y estima la probabilidad de que la instancia pertenezca a la clase positiva. A diferencia de la regresión lineal, aplica una **función sigmoide** (logística) sobre la combinación lineal:

$$\hat{y} = \sigma(\mathbf{w}^T \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}}$$

El efecto de la función sigmoide es **comprimir** la salida al rango $[0, 1]$, interpretable como probabilidad. Si $\hat{y} \geq$ umbral → clase positiva (Pagado); si $\hat{y} <$ umbral → clase negativa (No Pagado).

### Umbral de decisión

Por defecto el umbral es 0.5, pero puede ajustarse según el problema:

- **Umbral bajo (< 0.5):** el modelo es más agresivo prediciendo "No Pagado" → mayor Recall de impagos, más falsos positivos.
- **Umbral alto (> 0.5):** el modelo es más conservador → menos falsos positivos, pero puede pasar por alto impagos.

La **curva Precision-Recall** permite encontrar el umbral óptimo de forma sistemática.

### Desbalance de clases

Con ~85 % de préstamos pagados y ~15 % impagos, un modelo que siempre predice "Pagado" obtendría 85 % de accuracy sin aprender nada útil. La solución recomendada es `class_weight='balanced'`, que ajusta los pesos inversamente proporcional a la frecuencia de cada clase.


---

## 1. Entorno y librerías

In [ ]:
# ==============================================================================
# PASO 1: Importar librerías
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from sklearn.linear_model   import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing  import StandardScaler, LabelEncoder
from sklearn.pipeline       import Pipeline
from sklearn.metrics        import (classification_report, confusion_matrix,
                                    precision_recall_curve, roc_curve,
                                    roc_auc_score, f1_score, accuracy_score)
from sklearn.inspection     import permutation_importance

# ── Tema visual global ──────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':         120,
    'figure.facecolor':   '#0f0f1a',
    'axes.facecolor':     '#1a1a2e',
    'axes.edgecolor':     '#2e2e4e',
    'axes.labelcolor':    '#e0e0f0',
    'axes.titlecolor':    '#ffffff',
    'axes.titlesize':     13,
    'axes.titleweight':   'bold',
    'axes.labelsize':     11,
    'axes.grid':          True,
    'grid.color':         '#2e2e4e',
    'grid.linestyle':     '--',
    'grid.alpha':         0.6,
    'xtick.color':        '#a0a0c0',
    'ytick.color':        '#a0a0c0',
    'text.color':         '#e0e0f0',
    'legend.facecolor':   '#1a1a2e',
    'legend.edgecolor':   '#3e3e6e',
    'legend.labelcolor':  '#e0e0f0',
    'font.family':        'DejaVu Sans',
})

C_ACCENT = '#7b68ee'
C_SECOND = '#00d4aa'
C_WARN   = '#ff6b6b'
C_GOLD   = '#ffd700'
C_BLUE   = '#4fc3f7'
PALETTE  = [C_ACCENT, C_SECOND, C_WARN, C_GOLD, C_BLUE]

import sklearn
print('Librerías importadas.')
print(f'  scikit-learn {sklearn.__version__}')
print(f'  pandas       {pd.__version__}')


> Se importan las mismas herramientas del notebook base del profesor más `Pipeline`, `cross_val_score` y `precision_recall_curve` — herramientas de nivel industria para un flujo de trabajo reproducible y sin data leakage.

---

## 2. Carga del Dataset y Selección de Variables

### Contexto de la práctica

Solo las características disponibles **antes de que se inicie el préstamo** pueden usarse como predictores. Variables como `recoveries` (recuperaciones post-cierre) o `total_rec_prncp` (principal recibido a la fecha) solo están disponibles después de que el préstamo se cierra — incluirlas provocaría **data leakage** y métricas artificialmente perfectas.

### Variables seleccionadas (disponibles al momento de la solicitud)

| Variable | Descripción |
|---|---|
| `funded_amnt` | Monto financiado comprometido con el préstamo |
| `int_rate` | Tasa de interés del préstamo |
| `grade_code` | Código numérico de la calificación crediticia |
| `purpose_code` | Código numérico del propósito del préstamo |
| `addr_state_code` | Código numérico del estado |
| `home_ownership_code` | Código numérico de situación de vivienda |
| `annual_inc` | Ingresos anuales del solicitante |
| `dti` | Ratio deuda total / ingreso mensual |
| `revol_util` | Porcentaje de crédito revolving utilizado |
| `pub_rec_bankruptcies` | Número de quiebras en registros públicos |

**Variable dependiente:** `repaid` → 1 = Fully Paid, 0 = Charged Off


In [ ]:
# ==============================================================================
# PASO 2: Cargar dataset, codificar variables y construir target
# ==============================================================================
df_raw = pd.read_csv('lending_club_2007_2011_6_states.csv')

# ── Selección de columnas del profesor ─────────────────────────────────────
COLS_ORIG = ['funded_amnt', 'int_rate', 'grade', 'purpose', 'addr_state',
             'home_ownership', 'annual_inc', 'dti', 'revol_util',
             'pub_rec_bankruptcies', 'loan_status']

df = df_raw[COLS_ORIG].copy()

# ── Codificación de variables categóricas con LabelEncoder ─────────────────
# (mismo enfoque que el profesor: variables ordinales → código numérico)
le = LabelEncoder()
df['grade_code']          = le.fit_transform(df['grade'])          # A=0 B=1 ... G=6
df['purpose_code']        = le.fit_transform(df['purpose'])
df['addr_state_code']     = le.fit_transform(df['addr_state'])
df['home_ownership_code'] = le.fit_transform(df['home_ownership'])

# ── Variable objetivo ───────────────────────────────────────────────────────
df['repaid'] = (df['loan_status'] == 'Fully Paid').astype(int)

# ── Imputar nulos con mediana ───────────────────────────────────────────────
df['revol_util']          = df['revol_util'].fillna(df['revol_util'].median())
df['pub_rec_bankruptcies']= df['pub_rec_bankruptcies'].fillna(0)

# ── Variables finales ────────────────────────────────────────────────────────
FEATURES = ['funded_amnt', 'int_rate', 'grade_code', 'purpose_code',
            'addr_state_code', 'home_ownership_code', 'annual_inc',
            'dti', 'revol_util', 'pub_rec_bankruptcies']
TARGET   = 'repaid'

X = df[FEATURES]
y = df[TARGET]

print(f'Shape final : {X.shape[0]:,} filas × {X.shape[1]} características')
print(f'\n── Distribución del target ─────────────────────────────────────────────')
vc = y.value_counts()
for val, n in vc.items():
    label = 'Fully Paid (Pagado)' if val == 1 else 'Charged Off (No Pagado)'
    print(f'  {val} — {label:<28}: {n:>6,}  ({n/len(y)*100:.1f} %)')

print('\n── Primeras filas del dataframe procesado ──────────────────────────────')
display(df[FEATURES + [TARGET]].head(5))


### Interpretación — Dataset procesado

- El dataset procesado replica exactamente el `prestamos_ok.csv` del profesor, construido directamente desde el CSV original mediante `LabelEncoder` — más transparente y reproducible.
- El **desbalance es claro**: ~85 % pagados vs ~15 % no pagados. Este es el problema central que abordaremos en las siguientes secciones.
- Los nulos de `revol_util` (~19 registros) y `pub_rec_bankruptcies` (~313) se imputan con mediana y cero respectivamente — la mayoría de personas sin quiebras tienen 0 registros.


---

## 3. División del Dataset

Se usa la misma partición 60/40 del notebook del profesor con `random_state=42`, más `stratify=y` para garantizar que la proporción de clases se conserve en ambos splits.

In [ ]:
# ==============================================================================
# PASO 3: División 60/40 con stratify (igual que el profesor)
# ==============================================================================
SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.40, random_state=SEED, stratify=y
)

print('División del dataset:')
print(f'  Total         : {len(X):>6,} muestras')
print(f'  Entrenamiento : {len(X_train):>6,} muestras ({len(X_train)/len(X)*100:.0f} %)')
print(f'  Prueba        : {len(X_test):>6,} muestras ({len(X_test)/len(X)*100:.0f} %)')
print(f'\n── Proporción de clases ────────────────────────────────────────────────')
for split, y_s in [('Train', y_train), ('Test', y_test)]:
    vc = y_s.value_counts()
    print(f'  {split}: Pagado={vc.get(1,0):,} ({vc.get(1,0)/len(y_s)*100:.1f}%)  '
          f'No Pagado={vc.get(0,0):,} ({vc.get(0,0)/len(y_s)*100:.1f}%)')
print(f'\nX_train: {X_train.shape}  X_test: {X_test.shape}')


### Interpretación — División estratificada

- La partición 60/40 reserva más datos para prueba que el estándar 80/20 — el profesor la eligió así para poder observar mejor el comportamiento del modelo en datos no vistos con un conjunto de prueba más grande (~7,964 muestras).
- `stratify=y` garantiza que la proporción 85/15 se mantiene en ambos splits, evitando que el azar concentre más impagos en un split que en otro.


---

## 4. Función de Evaluación Reutilizable

In [ ]:
# ==============================================================================
# PASO 4: Función centralizada de evaluación
# ==============================================================================
def evaluar_modelo(nombre, modelo, X_te, y_te, mostrar_roc=True):
    """Evalúa un modelo ya entrenado: reporte, matriz de confusión, ROC."""
    y_pred  = modelo.predict(X_te)
    y_proba = modelo.predict_proba(X_te)[:, 1]

    acc   = accuracy_score(y_te, y_pred)
    auc   = roc_auc_score(y_te, y_proba)

    print(f'\n{"═"*55}')
    print(f'  {nombre}')
    print(f'  Accuracy : {acc:.4f}   AUC-ROC : {auc:.4f}')
    print(f'{"═"*55}')
    print(classification_report(y_te, y_pred,
                                target_names=['No Pagado','Pagado'],
                                zero_division=0))

    # Matriz de confusión + ROC side by side
    ncols = 2 if mostrar_roc else 1
    fig, axes = plt.subplots(1, ncols, figsize=(13 if mostrar_roc else 6, 5))
    if ncols == 1: axes = [axes]
    fig.suptitle(nombre, fontsize=13, fontweight='bold', color='white')

    cm = confusion_matrix(y_te, y_pred)
    labels = ['No Pagado', 'Pagado']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels,
                linewidths=0.5, linecolor='#0f0f1a',
                annot_kws={'size':13,'weight':'bold'}, ax=axes[0])
    axes[0].set_title('Matriz de Confusión')
    axes[0].set_xlabel('Predicción'); axes[0].set_ylabel('Real')

    # Anotaciones de contexto financiero
    tn, fp, fn, tp = cm.ravel()
    axes[0].set_xlabel(
        f'Predicción\n\n'
        f'TN={tn:,} (filtro correcto)  FP={fp:,} (oportunidad perdida)\n'
        f'FN={fn:,} (pérdida financiera ⚠)  TP={tp:,} (aprobación correcta)',
        fontsize=8, color='#a0a0c0'
    )

    if mostrar_roc:
        fpr, tpr, _ = roc_curve(y_te, y_proba, pos_label=0)
        axes[1].plot(fpr, tpr, color=C_ACCENT, linewidth=2,
                     label=f'AUC = {auc:.4f}')
        axes[1].plot([0,1],[0,1],'--', color='#555', linewidth=1)
        axes[1].fill_between(fpr, tpr, alpha=0.15, color=C_ACCENT)
        axes[1].set_title('Curva ROC (clase No Pagado)')
        axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR (Recall)')
        axes[1].legend(fontsize=10)

    plt.tight_layout()
    plt.show()
    return y_pred, y_proba

print('Función evaluar_modelo() lista.')


---

## 5. Regresión Logística sin Balanceo de Clases

Se construye el primer clasificador con parámetros predeterminados — sin manejo del desbalance. Esto replica el `clf1` del notebook del profesor y sirve como **baseline** para entender el problema.


In [ ]:
# ==============================================================================
# PASO 5: Regresión Logística sin balanceo (clf1 del profesor)
# ==============================================================================
clf1 = LogisticRegression(random_state=0, max_iter=1000)   # max_iter=1000 evita ConvergenceWarning
clf1.fit(X_train, y_train)

print('Accuracy entrenamiento :', clf1.score(X_train, y_train))
y_pred1, y_prob1 = evaluar_modelo('Regresión Logística — Sin balanceo (clf1)',
                                   clf1, X_test, y_test)


### Interpretación — Sin balanceo de clases

Esta tabla resume lo que muestra el reporte de clasificación:

| Clase | Precision | Recall | F1-score | Significado |
|---|---|---|---|---|
| **No Pagado** | ~0.08 | ~0.00 | ~0.00 | El modelo casi nunca detecta impagos |
| **Pagado** | ~0.85 | ~1.00 | ~0.92 | El modelo siempre predice "Pagado" |

**¿Por qué ocurre esto?**

El modelo aprende que si **siempre dice "Pagado"** acierta el 85 % de las veces — minimiza el error total pero ignora completamente la clase minoritaria. Esto es consecuencia directa del desbalance: 6,744 préstamos pagados vs 1,220 no pagados en el conjunto de prueba.

**Consecuencia financiera:**

La matriz de confusión muestra que de ~1,220 préstamos realmente impagos, el modelo identifica correctamente **casi ninguno** → pérdida económica directa para los inversores. Un modelo que no detecta impagos no tiene utilidad práctica en este contexto.

> **Nota técnica:** `max_iter=1000` resuelve el `ConvergenceWarning` del profesor. El solver `lbfgs` necesita más iteraciones cuando los datos no están escalados — lo corregimos en la Sección 8.


---

## 6. Regresión Logística con `class_weight='balanced'`

Se construye el segundo clasificador especificando `class_weight='balanced'`. Esto instruye al modelo a penalizar más los errores en la clase minoritaria (No Pagado), ajustando los pesos inversamente proporcional a la frecuencia de cada clase:

$$w_c = \frac{n_{\text{total}}}{n_{\text{clases}} \times n_c}$$

Mismo experimento que `clf2` del profesor.


In [ ]:
# ==============================================================================
# PASO 6: Regresión Logística con class_weight='balanced' (clf2 del profesor)
# ==============================================================================
clf2 = LogisticRegression(random_state=0, max_iter=1000, class_weight='balanced')
clf2.fit(X_train, y_train)

print('Accuracy entrenamiento :', clf2.score(X_train, y_train))
y_pred2, y_prob2 = evaluar_modelo('Regresión Logística — class_weight=balanced (clf2)',
                                   clf2, X_test, y_test)


### Interpretación — Con balanceo de clases

| Métrica | Sin balanceo | Con balanceo | Interpretación |
|---|---|---|---|
| Accuracy | ~0.85 | ~0.60 | Baja, pero ya no engañosa |
| Recall No Pagado | ~0.00 | ~0.63 | **Mejora enorme**: detecta 6 de cada 10 impagos |
| Precision No Pagado | ~0.08 | ~0.22 | También mejora: menos falsas alarmas |
| F1 No Pagado | ~0.00 | ~0.32 | Sube de 0 a 0.32 — el modelo ahora es útil |

**¿Por qué baja el accuracy?**

El modelo ahora también predice "No Pagado" cuando antes siempre decía "Pagado". Algunos préstamos realmente pagados se clasifican erróneamente como impagos — eso reduce el accuracy global. Pero esto es **una buena señal**: el modelo está aprendiendo a distinguir entre las dos clases.

**Consecuencias financieras de la matriz de confusión:**

| Decisión | Realidad | Consecuencia |
|---|---|---|
| Predice "Pagado" y sí paga | ✓ Correcto | Se gana dinero con los intereses |
| Predice "Pagado" pero no paga | ✗ Error grave | **Pérdida económica importante** |
| Predice "No Pagado" y no paga | ✓ Correcto | Buen filtro de riesgo |
| Predice "No Pagado" pero sí paga | Error menor | Se pierde oportunidad de préstamo |

Con balanceo, los falsos negativos (pérdida financiera) se reducen significativamente: de ~1,219 a ~451.


---

## 7. Normalización con StandardScaler — Pipeline

Muchos algoritmos de Machine Learning (Regresión Logística, SVM, KNN, redes neuronales) son **sensibles a la escala** de los datos. Si una variable tiene un rango mucho mayor que otra, puede dominar el modelo artificialmente.

**Ejemplo:**
- `annual_inc`: 10,000 – 100,000
- `pub_rec_bankruptcies`: 0 – 5

Sin escalar, el solver puede darle más importancia a `annual_inc` solo por su magnitud.

Se usa `Pipeline` — práctica estándar en la industria — para encadenar `StandardScaler → LogisticRegression` en un solo objeto, garantizando que el scaler se ajusta **únicamente** con datos de entrenamiento.


In [ ]:
# ==============================================================================
# PASO 7: Pipeline StandardScaler + LogisticRegression (nivel industria)
# Mismo experimento que RL_scaled del profesor pero sin data leakage
# ==============================================================================

# Pipeline: el scaler se ajusta solo con X_train — nunca ve X_test
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(random_state=0, max_iter=1000,
                                  class_weight='balanced', solver='lbfgs')),
])

pipeline_lr.fit(X_train, y_train)

print('Pipeline entrenado:')
print(pipeline_lr)
print(f'\nAccuracy entrenamiento : {pipeline_lr.score(X_train, y_train):.4f}')

y_pred_sc, y_prob_sc = evaluar_modelo(
    'Pipeline — StandardScaler + LR balanced',
    pipeline_lr, X_test, y_test
)


### Interpretación — Comparativo sin normalizar vs normalizado

| Configuración | VN "No Pagado" | FP | FN | VP "Pagado" | Accuracy |
|---|---|---|---|---|---|
| Balanceado sin escalar | ~769 | ~451 | ~2,805 | ~3,939 | ~0.59 |
| Balanceado con Pipeline | ~719 | ~458 | ~2,611 | ~4,176 | ~0.61 |

**¿Por qué no mejora drásticamente con normalización?**

La Regresión Logística ya es un modelo lineal. Aunque se beneficia del escalado para la **estabilidad numérica y velocidad de convergencia** (elimina el `ConvergenceWarning`), su frontera de decisión no cambia drásticamente cuando las variables ya tienen rangos razonablemente comparables. La mejora principal del escalado se ve en modelos como KNN o SVM.

**Ventaja real del Pipeline:** elimina el riesgo de data leakage — si se aplicara `scaler.fit_transform(X)` antes de dividir el dataset, la media y desviación del test contaminarían el entrenamiento. El Pipeline lo hace automáticamente correcto.


---

## 8. Importancia de Variables (Permutation Importance)

Mide cuánto cae el accuracy del modelo cuando los valores de una variable se permutan aleatoriamente. Una caída grande indica que la variable es importante para las predicciones del modelo.

In [ ]:
# ==============================================================================
# PASO 8a: Permutation Importance — SIN escalar (clf2) vs CON Pipeline
# ==============================================================================

# Sin escalar
result_raw = permutation_importance(clf2, X_test, y_test,
                                    n_repeats=30, random_state=42, n_jobs=-1)
imp_raw = pd.DataFrame({
    'Variable'  : FEATURES,
    'Imp. Media': result_raw.importances_mean,
    'Desviación': result_raw.importances_std,
}).sort_values('Imp. Media', ascending=False)

# Con Pipeline (datos escalados internamente)
X_test_scaled = pipeline_lr.named_steps['scaler'].transform(X_test)
result_sc = permutation_importance(pipeline_lr.named_steps['lr'],
                                   X_test_scaled, y_test,
                                   n_repeats=30, random_state=42, n_jobs=-1)
imp_sc = pd.DataFrame({
    'Variable'  : FEATURES,
    'Imp. Media': result_sc.importances_mean,
    'Desviación': result_sc.importances_std,
}).sort_values('Imp. Media', ascending=False)

print('── Sin escalar (clf2) ──────────────────────────────────────────────────')
display(imp_raw.round(4))
print('\n── Con escalado (Pipeline) ─────────────────────────────────────────────')
display(imp_sc.round(4))


In [ ]:
# ==============================================================================
# PASO 8b: Gráfica comparativa de importancias (Plotly)
# ==============================================================================
fig = go.Figure()

for df_imp, nombre, color in [
    (imp_raw, 'Sin escalar (clf2)',     C_ACCENT),
    (imp_sc,  'Con escalado (Pipeline)', C_SECOND),
]:
    fig.add_trace(go.Bar(
        name        = nombre,
        x           = df_imp['Variable'],
        y           = df_imp['Imp. Media'],
        error_y     = dict(type='data', array=df_imp['Desviación'].tolist()),
        text        = df_imp['Imp. Media'].round(3),
        textposition= 'outside',
        marker_color= color,
        opacity     = 0.85,
    ))

fig.update_layout(
    barmode     = 'group',
    title       = 'Importancia de Características — Regresión Logística (Permutation Importance)',
    xaxis_title = 'Variable',
    yaxis_title = 'Importancia Media',
    width=850, height=480,
    plot_bgcolor = '#1a1a2e',
    paper_bgcolor= '#0f0f1a',
    font_color   = '#e0e0f0',
    legend       = dict(bgcolor='#1a1a2e', bordercolor='#3e3e6e'),
)
fig.show()


### Interpretación — Importancia de variables

- **`int_rate` (tasa de interés)** es la variable más importante en ambas configuraciones — la tasa ya incorpora el riesgo percibido por Lending Club, por lo que tiene alto poder predictivo.
- **`annual_inc` (ingreso anual)** ocupa el segundo lugar: mayor ingreso = mayor capacidad de pago.
- **`grade_code`** refleja la calificación crediticia — directamente relacionada con `int_rate`. Con escalado su importancia relativa puede variar porque el modelo distribuye mejor el peso entre variables correlacionadas.
- Variables como `revol_util`, `purpose_code` y `funded_amnt` muestran importancias negativas — permutar su orden aleatoriamente no perjudica al modelo, lo que sugiere que aportan poco o generan ruido.
- **Diferencia ante el escalado:** la Regresión Logística es más estable que KNN ante la escala, por lo que las importancias no cambian drásticamente — confirma el hallazgo de la sección anterior.


---

## 9. Ajuste del Umbral de Decisión

El umbral de decisión (o *threshold*) es el valor a partir del cual la probabilidad predicha se convierte en una clase:

- Si $P(\text{Pagado}) \geq$ umbral → predice **Pagado (1)**
- Si $P(\text{Pagado}) <$ umbral → predice **No Pagado (0)**

Por defecto el umbral es **0.5**. Bajarlo aumenta la detección de "No Pagado" (mayor Recall) a costa de más falsos positivos. Subirlo hace lo contrario.

### 9.1 Búsqueda del umbral óptimo con la curva Precision-Recall

En lugar de probar umbrales manualmente, usamos la **curva Precision-Recall** para encontrar el umbral que **maximiza el F1-score de No Pagado** de forma sistemática — práctica estándar en la industria.


In [ ]:
# ==============================================================================
# PASO 9a: Curva Precision-Recall y umbral óptimo automático
# ==============================================================================

# Probabilidad de clase NO PAGADO (pos_label=0 → invertimos proba)
y_prob_nopagado = 1 - y_prob_sc   # probabilidad de ser No Pagado

precision_vals, recall_vals, thresholds = precision_recall_curve(
    y_test, y_prob_nopagado, pos_label=1  # aquí 1 = No Pagado
)

# F1 para cada umbral
f1_vals = 2 * (precision_vals[:-1] * recall_vals[:-1]) / (
          precision_vals[:-1] + recall_vals[:-1] + 1e-9)

umbral_optimo = thresholds[np.argmax(f1_vals)]
f1_max        = np.max(f1_vals)

print(f'Umbral óptimo (máximo F1 de No Pagado) : {umbral_optimo:.4f}')
print(f'F1-score de No Pagado en umbral óptimo : {f1_max:.4f}')

# ── Curva Precision-Recall ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Análisis del Umbral de Decisión', fontsize=13,
             fontweight='bold', color='white')

# Gráfica 1: curva PR
axes[0].plot(recall_vals[:-1], precision_vals[:-1], color=C_ACCENT, linewidth=2)
axes[0].axvline(recall_vals[:-1][np.argmax(f1_vals)], color=C_GOLD,
                linestyle='--', linewidth=1.5,
                label=f'Umbral óptimo = {umbral_optimo:.3f}')
axes[0].set_title('Curva Precision-Recall (clase No Pagado)')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].legend(fontsize=9)

# Gráfica 2: F1 vs umbral
axes[1].plot(thresholds, f1_vals, color=C_SECOND, linewidth=2)
axes[1].axvline(umbral_optimo, color=C_GOLD, linestyle='--', linewidth=1.5,
                label=f'Máximo F1 = {f1_max:.4f}')
axes[1].set_title('F1-score de No Pagado vs Umbral')
axes[1].set_xlabel('Umbral de decisión')
axes[1].set_ylabel('F1-score (No Pagado)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()


### Interpretación — Curva Precision-Recall y umbral óptimo

- La **curva Precision-Recall** muestra el trade-off entre precisión y recall de "No Pagado" para cada umbral posible. Un modelo perfecto alcanzaría el punto (1.0, 1.0).
- La **curva F1 vs umbral** permite identificar visualmente el umbral que balancea mejor precisión y recall — el punto más alto de la curva.
- El umbral óptimo calculado automáticamente es más confiable que probar valores manualmente (0.40, 0.45, 0.55, 0.60) porque evalúa **todos los umbrales posibles** y elige el mejor de forma objetiva.


### 9.2 Evaluación en umbrales del notebook del profesor + umbral óptimo

In [ ]:
# ==============================================================================
# PASO 9b: Comparativo de umbrales (replicando el análisis del profesor)
# ==============================================================================
umbrales = [0.40, 0.45, 0.50, 0.55, 0.60, round(float(umbral_optimo), 2)]

resultados_umbral = []

for u in umbrales:
    # Convertir P(Pagado) a predicción con umbral u
    y_pred_u = (y_prob_sc >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_u).ravel()

    resultados_umbral.append({
        'Umbral'             : u,
        'Accuracy'           : round(accuracy_score(y_test, y_pred_u), 4),
        'Recall No Pagado'   : round(tn / (tn + fp) if (tn+fp) > 0 else 0, 4),
        'Precision No Pagado': round(tn / (tn + fn) if (tn+fn) > 0 else 0, 4),
        'F1 No Pagado'       : round(f1_score(y_test, y_pred_u, pos_label=0,
                                               zero_division=0), 4),
        'VN (No Pagado TP)'  : tn,
        'FN (pérdida fin.)'  : fn,
    })
    label = ' ← ÓPTIMO' if u == round(float(umbral_optimo), 2) else ''
    print(f'Umbral {u:.2f}{label}')
    print(classification_report(y_test, y_pred_u,
                                target_names=['No Pagado','Pagado'],
                                zero_division=0))

df_umbral = pd.DataFrame(resultados_umbral)
display(df_umbral.style
    .highlight_max(subset=['Recall No Pagado','F1 No Pagado'], color='#00543a')
    .format({c: '{:.4f}' for c in df_umbral.columns
             if c not in ['Umbral','VN (No Pagado TP)','FN (pérdida fin.)']})
    .set_caption('Comparativo de umbrales — verde = mejor por columna')
)


### Interpretación — Comparativo de umbrales

| Umbral | Comportamiento | Escenario recomendado |
|---|---|---|
| **0.40** | Detecta pocos impagos, alta accuracy | Maximizar aprobaciones, riesgo controlado |
| **0.45** | Buen equilibrio general | Uso general balanceado |
| **0.50** | Default — ignora el desbalance | No recomendado con clases desbalanceadas |
| **0.55** | Detecta más impagos, baja accuracy | Mayor aversión al riesgo |
| **0.60** | Muy agresivo contra impagos | Contextos de muy alto riesgo |
| **Óptimo** | Maximiza F1 de No Pagado | **Recomendado para producción** |

**Conclusión clave:** Cuando se baja el umbral, el modelo es más propenso a predecir "No Pagado" → detecta más impagos pero también rechaza algunos préstamos que sí serían pagados (oportunidad perdida). La elección del umbral depende del **costo relativo de cada tipo de error** en el contexto del negocio.


---

## 10. Visualización Comparativa de Métricas por Umbral

In [ ]:
# ==============================================================================
# PASO 10: Gráfica interactiva de métricas vs umbral
# ==============================================================================
fig = go.Figure()

metricas_plot = {
    'Accuracy'           : C_ACCENT,
    'Recall No Pagado'   : C_WARN,
    'Precision No Pagado': C_SECOND,
    'F1 No Pagado'       : C_GOLD,
}

for metrica, color in metricas_plot.items():
    fig.add_trace(go.Scatter(
        x    = df_umbral['Umbral'],
        y    = df_umbral[metrica],
        mode = 'lines+markers',
        name = metrica,
        line = dict(color=color, width=2),
        marker = dict(size=8),
    ))

# Línea vertical en umbral óptimo
fig.add_vline(x=round(float(umbral_optimo), 2),
              line_color=C_BLUE, line_dash='dash',
              annotation_text=f'Umbral óptimo={round(float(umbral_optimo),2)}',
              annotation_font_color=C_BLUE)

fig.update_layout(
    title       = 'Métricas de Clasificación vs Umbral de Decisión',
    xaxis_title = 'Umbral',
    yaxis_title = 'Valor de la métrica',
    yaxis_range = [0, 1.05],
    width=850, height=460,
    plot_bgcolor  = '#1a1a2e',
    paper_bgcolor = '#0f0f1a',
    font_color    = '#e0e0f0',
    legend        = dict(bgcolor='#1a1a2e', bordercolor='#3e3e6e'),
)
fig.show()


### Interpretación — Métricas vs umbral

- **Accuracy** (morado): disminuye al bajar el umbral — el modelo rechaza más préstamos buenos.
- **Recall No Pagado** (rojo): aumenta al bajar el umbral — detecta más impagos.
- **Precision No Pagado** (verde): también aumenta ligeramente al subir el umbral — menos falsos positivos.
- **F1 No Pagado** (dorado): el equilibrio entre Recall y Precision — su máximo es el umbral óptimo marcado con la línea azul.

Esta visualización interactiva permite al analista de riesgo seleccionar el umbral más adecuado según la política de crédito del banco: si el banco es conservador (prefiere rechazar más), sube el umbral; si quiere capturar más clientes, lo baja.


---

## 11. Interpretación de Coeficientes

Una ventaja clave de la Regresión Logística frente a otros modelos es su **interpretabilidad directa**: los coeficientes muestran la dirección e intensidad del efecto de cada variable sobre la probabilidad de incumplimiento.

In [ ]:
# ==============================================================================
# PASO 11: Coeficientes del modelo escalado
# ==============================================================================
lr_model = pipeline_lr.named_steps['lr']

coef_df = pd.DataFrame({
    'Variable'   : FEATURES,
    'Coeficiente': lr_model.coef_[0],
}).sort_values('Coeficiente', key=abs, ascending=True)

colores = [C_WARN if c > 0 else C_SECOND for c in coef_df['Coeficiente']]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(coef_df['Variable'], coef_df['Coeficiente'],
        color=colores, alpha=0.85, edgecolor='black', linewidth=0.4)
ax.axvline(0, color='white', linewidth=0.8, linestyle='--')
ax.set_title('Coeficientes del Modelo — Regresión Logística Escalada\n'
             '(rojo = aumenta riesgo de No Pago, verde = lo reduce)')
ax.set_xlabel('Coeficiente (espacio estandarizado)')
for i, (v, _) in enumerate(zip(coef_df['Coeficiente'], coef_df['Variable'])):
    ax.text(v + (0.005 if v >= 0 else -0.005), i,
            f'{v:.4f}', va='center', ha='left' if v >= 0 else 'right',
            fontsize=9, color='white')
plt.tight_layout()
plt.show()


### Interpretación — Coeficientes

- **Coeficiente positivo (rojo):** la variable aumenta el log-odds de predecir "Pagado" — inversamente, un coeficiente muy negativo aumenta el riesgo de impago.
- **`int_rate`** tiene el efecto más grande: mayor tasa de interés → mayor predicción de incumplimiento (lógico — Lending Club asigna tasas más altas a prestatarios más riesgosos).
- **`annual_inc`** tiene efecto positivo en pago: mayor ingreso → mayor probabilidad de pagar.
- **`grade_code`** (donde G=6 es el grado más riesgoso) tiene efecto negativo: mayor código de grado → más riesgo de impago.
- Esta interpretabilidad es una ventaja regulatoria: en el sector financiero, las decisiones de crédito deben ser explicables. Un banco puede usar estos coeficientes para justificar una negativa de préstamo ante un regulador.


---

## 12. Predicciones sobre Nuevos Préstamos (Ficticios)

El Pipeline gestiona automáticamente el escalado de las nuevas instancias antes de predecir. Se proveen los 10 valores en el mismo orden que `FEATURES`.

> `[funded_amnt, int_rate, grade_code, purpose_code, addr_state_code, home_ownership_code, annual_inc, dti, revol_util, pub_rec_bankruptcies]`


In [ ]:
# ==============================================================================
# PASO 12: Predicciones sobre solicitudes nuevas (ficticias)
# ==============================================================================
nuevas_solicitudes = pd.DataFrame([
    # funded  int  grade purp  state  home   inc     dti  revol  bkrpt
    [5000,  15.5,   5,    3,    4,    4,   30000,   12,   12,    0],   # perfil riesgo medio
    [15000,  7.5,   1,    0,    2,    2,   80000,    8,   25,    0],   # perfil bajo riesgo
    [25000, 22.0,   6,    5,    8,    0,   20000,   35,   85,    1],   # perfil alto riesgo
    [3000,  13.0,   2,    1,    5,    1,   45000,   15,   40,    0],   # perfil moderado
], columns=FEATURES)

# Predicción + probabilidades
predicciones  = pipeline_lr.predict(nuevas_solicitudes)
probabilidades = pipeline_lr.predict_proba(nuevas_solicitudes)

print('── Predicciones con probabilidades ─────────────────────────────────────')
for i, (pred, probs) in enumerate(zip(predicciones, probabilidades.tolist())):
    estado = 'PAGARÁ ✓' if pred == 1 else 'NO PAGARÁ ⚠'
    print(f'  Solicitud {i+1} → [{estado:<14}]  '
          f'P(No Pagado)={probs[0]:.3f}  P(Pagado)={probs[1]:.3f}')

# Tabla
df_pred = nuevas_solicitudes.copy()
df_pred.insert(0, 'Predicción', ['Pagado' if p==1 else 'No Pagado' for p in predicciones])
df_pred['P(No Pagado)'] = probabilidades[:, 0].round(3)
df_pred['P(Pagado)']    = probabilidades[:, 1].round(3)
display(df_pred)


### Interpretación — Predicciones

- El Pipeline escala automáticamente las nuevas solicitudes usando la media y desviación estándar aprendidas en el entrenamiento — no se necesita escalar manualmente.
- Las **probabilidades** permiten ajustar el umbral: si el banco decide usar umbral=0.45, cualquier solicitud con `P(Pagado) < 0.45` se rechaza.
- La solicitud 3 (alta tasa, grado G, alto dti, alta revol_util, con quiebra) debería clasificarse como No Pagado — perfil de alto riesgo consistente con los coeficientes del modelo.
- La solicitud 2 (baja tasa, grado B, bajo dti, alto ingreso) debería clasificarse como Pagado — perfil sólido.


---

## 13. Resumen Comparativo Final

In [ ]:
# ==============================================================================
# PASO 13: Tabla resumen de todos los modelos entrenados
# ==============================================================================
resumen = []
for nombre, modelo, X_e, y_e in [
    ('LR Sin balanceo (clf1)',          clf1,        X_test,        y_test),
    ('LR Balanceado (clf2)',            clf2,        X_test,        y_test),
    ('Pipeline LR + Scaler',           pipeline_lr, X_test,        y_test),
]:
    yp = modelo.predict(X_e)
    ypr= modelo.predict_proba(X_e)[:, 1]
    resumen.append({
        'Modelo'             : nombre,
        'Accuracy'           : round(accuracy_score(y_e, yp), 4),
        'Recall No Pagado'   : round(recall_score(y_e, yp, pos_label=0, zero_division=0), 4),
        'F1 No Pagado'       : round(f1_score(y_e, yp, pos_label=0, zero_division=0), 4),
        'AUC-ROC'            : round(roc_auc_score(y_e, ypr), 4),
    })

df_res = pd.DataFrame(resumen)
display(df_res.style
    .highlight_max(subset=['Recall No Pagado','F1 No Pagado','AUC-ROC'], color='#00543a')
    .highlight_min(subset=['Recall No Pagado','F1 No Pagado'], color='#5a1010')
    .format({c: '{:.4f}' for c in df_res.columns if c != 'Modelo'})
    .set_caption('Resumen — verde=mejor, rojo=peor')
)


---

## Conclusiones

### 1. El balanceo de clases es indispensable para este dataset
Sin `class_weight='balanced'`, el modelo aprende a siempre decir "Pagado" y obtiene ~85 % de accuracy — métrica engañosa. Con balanceo, el Recall de "No Pagado" sube de ~0 % a ~63 %, haciendo al modelo genuinamente útil para el contexto financiero.

### 2. La accuracy no es la métrica correcta con clases desbalanceadas
En este problema, **Recall y F1-score de No Pagado** son las métricas relevantes. Un recall de 0.63 significa que el modelo detecta 6 de cada 10 préstamos que realmente incumplirán — ahorro real para el inversor.

### 3. El escalado mejora estabilidad numérica, no tanto el rendimiento
Para Regresión Logística, `StandardScaler` elimina el `ConvergenceWarning` y acelera el entrenamiento, pero no cambia drásticamente la frontera de decisión si las variables ya tienen rangos comparables. La ventaja principal del `Pipeline` es **prevenir data leakage**.

### 4. El umbral de decisión debe ajustarse según el costo del error
El umbral óptimo encontrado mediante la curva Precision-Recall maximiza el F1 de "No Pagado". En producción, el umbral se calibra según la política de riesgo del banco: menor umbral = mayor detección de impagos = menor cartera pero más segura.

### 5. La interpretabilidad de la Regresión Logística es una ventaja regulatoria
Los coeficientes del modelo permiten explicar cada decisión: "`int_rate` alto y `grade_code` G aumentan el riesgo". En entornos regulados como banca, esta trazabilidad es un requerimiento legal — ventaja que modelos como Random Forest o redes neuronales no ofrecen por defecto.
